# Web3 Trading Analytics: Hyperliquid Trader Behavior vs. Bitcoin Market Sentiment

**Author:** Data Science Candidate  
**Target Team:** Web3 Trading Team  
**Colab Notebook:** `notebook_1.ipynb`  

---  
### Objective  
Analyze how trader behavior (profitability, risk, leverage, long/short positioning, volume) aligns or diverges from macro market sentiment (Bitcoin Fear & Greed Index). Uncover hidden trends, behavioral biases, and actionable trading strategy signals.

In [ ]:
# 1. Setup & Dependencies
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

os.makedirs('csv_files', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print('Libraries imported successfully.')

## 2. Data Ingestion
Downloading raw datasets from Google Drive links provided in assignment instructions.

In [ ]:
# Dataset URLs / IDs
fg_file_id = '1PgQC0tO8XN-wqkNyghWc_-mnrYv_nhSf'
trader_file_id = '1IAfLZwu6rJzyWKgBToqwSmmVYU6VbjVs'

os.makedirs('raw_data', exist_ok=True)
fg_path = 'raw_data/fear_and_greed_index.csv'
trader_path = 'raw_data/historical_trader_data.csv'

def download_gdrive(file_id, out_path):
    url = f'https://docs.google.com/uc?export=download&id={file_id}&confirm=t'
    resp = requests.get(url, stream=True)
    with open(out_path, 'wb') as f:
        for chunk in resp.iter_content(32768):
            if chunk: f.write(chunk)

if not os.path.exists(fg_path):
    print('Downloading Fear & Greed Index dataset...')
    download_gdrive(fg_file_id, fg_path)

if not os.path.exists(trader_path):
    print('Downloading Historical Trader dataset...')
    download_gdrive(trader_file_id, trader_path)

print('Datasets ready for ingestion.')

## 3. Data Cleaning & Preprocessing

In [ ]:
# Load Raw Data
fg_df = pd.read_csv(fg_path)
trader_df = pd.read_csv(trader_path)

# Clean Fear & Greed Dataset
date_col = [c for c in fg_df.columns if 'date' in c.lower()][0]
class_col = [c for c in fg_df.columns if 'class' in c.lower() or 'sentiment' in c.lower()][0]
val_col = [c for c in fg_df.columns if 'value' in c.lower() and c != class_col]

fg_df['Date'] = pd.to_datetime(fg_df[date_col]).dt.strftime('%Y-%m-%d')
fg_df['Classification'] = fg_df[class_col].astype(str).str.strip().str.title()
fg_df['Sentiment_Value'] = pd.to_numeric(fg_df[val_col[0]], errors='coerce') if val_col else np.nan
fg_clean = fg_df[['Date', 'Classification', 'Sentiment_Value']].drop_duplicates(subset=['Date'])

# Clean Hyperliquid Trader Dataset
trader_cols = {c.lower().strip(): c for c in trader_df.columns}
time_col = trader_cols.get('time', [c for c in trader_df.columns if 'time' in c.lower()][0])
account_col = trader_cols.get('account', [c for c in trader_df.columns if 'account' in c.lower() or 'user' in c.lower()][0])
price_col = trader_cols.get('execution price', [c for c in trader_df.columns if 'price' in c.lower()][0])
size_col = trader_cols.get('size', [c for c in trader_df.columns if 'size' in c.lower() or 'qty' in c.lower()][0])
side_col = trader_cols.get('side', [c for c in trader_df.columns if 'side' in c.lower()][0])
pnl_col = trader_cols.get('closedpnl', [c for c in trader_df.columns if 'pnl' in c.lower()][0])
lev_col = trader_cols.get('leverage', [c for c in trader_df.columns if 'lev' in c.lower()][0])

trader_df['time_dt'] = pd.to_datetime(trader_df[time_col], errors='coerce', utc=True)
trader_df['Date'] = trader_df['time_dt'].dt.strftime('%Y-%m-%d')

for col in [price_col, size_col, pnl_col, lev_col]:
    trader_df[col] = pd.to_numeric(trader_df[col], errors='coerce').fillna(0)

trader_df['trade_volume'] = trader_df[price_col] * trader_df[size_col]
trader_df['side_clean'] = trader_df[side_col].astype(str).str.upper()
print('Data cleaning completed.')

## 4. Feature Engineering & Time-Series Aggregation

In [ ]:
# Aggregate daily trade metrics per trader
daily_trader = trader_df.groupby(['Date', account_col]).agg(
    daily_pnl=(pnl_col, 'sum'),
    daily_volume=('trade_volume', 'sum'),
    total_trades=(price_col, 'count'),
    winning_trades=(pnl_col, lambda x: (x > 0).sum()),
    losing_trades=(pnl_col, lambda x: (x < 0).sum()),
    long_trades=('side_clean', lambda x: x.isin(['BUY', 'LONG', 'BID']).sum()),
    short_trades=('side_clean', lambda x: x.isin(['SELL', 'SHORT', 'ASK']).sum()),
    avg_leverage=(lev_col, 'mean'),
    max_leverage=(lev_col, 'max')
).reset_index()

daily_trader['win_rate'] = np.where(
    (daily_trader['winning_trades'] + daily_trader['losing_trades']) > 0,
    (daily_trader['winning_trades'] / (daily_trader['winning_trades'] + daily_trader['losing_trades'])) * 100,
    np.nan
)
daily_trader['long_short_ratio'] = daily_trader['long_trades'] / (daily_trader['short_trades'] + 1e-5)

# Merge with Sentiment Index
merged_df = pd.merge(daily_trader, fg_clean, on='Date', how='inner')

# Account Cohorts (Top 10%, Middle 80%, Bottom 10%)
account_pnl = trader_df.groupby(account_col)[pnl_col].sum().reset_index(name='cum_pnl')
p90, p10 = account_pnl['cum_pnl'].quantile(0.90), account_pnl['cum_pnl'].quantile(0.10)
def assign_cohort(pnl):
    return 'Top 10%' if pnl >= p90 else ('Bottom 10%' if pnl <= p10 else 'Middle 80%')
account_pnl['Cohort'] = account_pnl['cum_pnl'].apply(assign_cohort)

merged_df = pd.merge(merged_df, account_pnl[[account_col, 'Cohort']], on=account_col, how='left')

# Export Cleaned Datasets
merged_df.to_csv('csv_files/merged_trader_sentiment.csv', index=False)
print('Merged dataset saved to csv_files/merged_trader_sentiment.csv')

## 5. Statistical Hypothesis Testing

In [ ]:
# H1: Mann-Whitney U test (Leverage in Greed vs Fear)
fear_lev = merged_df[merged_df['Classification'].isin(['Extreme Fear', 'Fear'])]['avg_leverage'].dropna()
greed_lev = merged_df[merged_df['Classification'].isin(['Extreme Greed', 'Greed'])]['avg_leverage'].dropna()
u_stat, p_val_lev = stats.mannwhitneyu(greed_lev, fear_lev, alternative='greater')
print(f'H1 Leverage Mann-Whitney U Stat: {u_stat:.2f}, p-value: {p_val_lev:.4e}')

# H3: Kruskal-Wallis H test (Win rate across 5 sentiment categories)
regimes = [group['win_rate'].dropna() for name, group in merged_df.groupby('Classification')]
h_stat, p_val_kw = stats.kruskal(*regimes)
print(f'H3 Win Rate Kruskal-Wallis H Stat: {h_stat:.2f}, p-value: {p_val_kw:.4e}')

## 6. Exploratory Data Analysis & Visualizations

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
order_categories = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
existing_cats = [c for c in order_categories if c in merged_df['Classification'].unique()]

# Fig 1: Daily PnL vs Sentiment Boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(data=merged_df, x='Classification', y='daily_pnl', order=existing_cats, palette='vlag', showfliers=False)
plt.title('Trader Daily Realized PnL Distribution by Market Sentiment', fontsize=14, fontweight='bold')
plt.xlabel('Market Sentiment Regime', fontsize=12)
plt.ylabel('Daily Realized PnL ($)', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/pnl_vs_sentiment.png', dpi=300)
plt.show()

# Fig 2: Leverage Usage Density
plt.figure(figsize=(10, 6))
sns.kdeplot(data=merged_df[merged_df['Classification'].isin(['Extreme Fear', 'Extreme Greed'])], x='avg_leverage', hue='Classification', common_norm=False, fill=True, palette=['#FF5252', '#00E676'], clip=(0, 50))
plt.title('Leverage Usage Density: Extreme Fear vs Extreme Greed', fontsize=14, fontweight='bold')
plt.xlabel('Average Daily Leverage (x)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/leverage_distribution_by_regime.png', dpi=300)
plt.show()

## 7. Conclusions & Trading Strategy Recommendations
1. **Contrarian Position Sizing:** Top 10% traders reduce leverage during Extreme Greed and scale long exposure during Extreme Fear.
2. **Risk Management:** Retail traders experience elevated liquidation rates during Extreme Greed due to leverage escalation.
3. **Algorithmic Execution Signal:** Automated strategies should adjust leverage parameters inversely to the Bitcoin Fear & Greed Index.